In [43]:
import warnings
warnings.filterwarnings('ignore')

import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

In [44]:
train_df=pd.read_parquet('/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/interim/final_train.parquet')
test_df=pd.read_parquet('/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/interim/final_test.parquet')

In [45]:

# ============================================================
# CONFIG
# ============================================================

TARGET_COL = 'TARGET'
ID_COL = 'SK_ID_CURR'

N_SPLITS = 5
RANDOM_STATE = 42
str_cols = train_df.select_dtypes(
    include=['object', 'string']
).columns

print(str_cols)

for col in str_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')


Index(['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY'], dtype='str')


In [46]:

# ============================================================
# FEATURES
# ============================================================

features = [
    col for col in train_df.columns
    if col not in [TARGET_COL, ID_COL]
]

print(f"Number of Features: {len(features)}")

X = train_df[features]
y = train_df[TARGET_COL]

X_test = test_df[features]



Number of Features: 1043


In [47]:

# ============================================================
# CATEGORICAL FEATURES
# ============================================================

cat_cols = X.select_dtypes(
    include=['category']
).columns.tolist()

print(f"Categorical Features: {len(cat_cols)}")


Categorical Features: 18


In [48]:


# ============================================================
# LIGHTGBM PARAMETERS
# ============================================================
params = {
    'objective': 'binary',
    'metric': 'auc',

    'boosting_type': 'gbdt',

    'learning_rate': 0.02,
    'n_estimators': 15000,

    'num_leaves': 128,
    'max_depth': -1,

    'min_child_samples': 50,

    'subsample': 0.9,
    'subsample_freq': 1,

    'colsample_bytree': 0.9,

    'reg_alpha': 0.1,
    'reg_lambda': 1.0,

    'random_state': 42,
    'n_jobs': -1,

    'verbosity': -1
}



In [49]:

# ============================================================
# CV SETUP
# ============================================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)



In [50]:

# ============================================================
# STORAGE
# ============================================================

oof_preds = np.zeros(len(train_df))

test_preds = np.zeros(len(test_df))

cv_scores = []

feature_importance = pd.DataFrame()


In [51]:


# ============================================================
# TRAINING LOOP
# ============================================================
# original parameters

for fold, (train_idx, valid_idx) in enumerate(
        skf.split(X, y), 1):

    print("=" * 60)
    print(f"FOLD {fold}")
    print("=" * 60)

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    model = lgb.LGBMClassifier(
        **params
    )

    model.fit(
        X_train,
        y_train,

        eval_set=[
            (X_valid, y_valid)
        ],

        eval_metric='auc',

        categorical_feature=cat_cols,

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=300,
                verbose=True
            ),
            lgb.log_evaluation(
                period=200
            )
        ]
    )

    # ========================================================
    # VALIDATION PREDICTIONS
    # ========================================================

    valid_preds = model.predict_proba(
        X_valid,
        num_iteration=model.best_iteration_
    )[:, 1]

    oof_preds[valid_idx] = valid_preds

    # ========================================================
    # TEST PREDICTIONS
    # ========================================================

    fold_test_preds = model.predict_proba(
        X_test,
        num_iteration=model.best_iteration_
    )[:, 1]

    test_preds += fold_test_preds / N_SPLITS

    # ========================================================
    # FOLD SCORE
    # ========================================================

    fold_auc = roc_auc_score(
        y_valid,
        valid_preds
    )

    cv_scores.append(fold_auc)

    print(
        f"Fold {fold} AUC = {fold_auc:.6f}"
    )

    # ========================================================
    # FEATURE IMPORTANCE
    # ========================================================

    fold_importance = pd.DataFrame({
        'feature': features,
        'importance': model.feature_importances_,
        'fold': fold
    })

    feature_importance = pd.concat(
        [feature_importance, fold_importance],
        axis=0,
        ignore_index=True
    )

    del (
        X_train,
        X_valid,
        y_train,
        y_valid,
        valid_preds,
        fold_test_preds
    )

    gc.collect()
    


FOLD 1
Training until validation scores don't improve for 300 rounds
[200]	valid_0's auc: 0.768766
[400]	valid_0's auc: 0.77444
[600]	valid_0's auc: 0.775017
[800]	valid_0's auc: 0.775453
[1000]	valid_0's auc: 0.77517
Early stopping, best iteration is:
[830]	valid_0's auc: 0.775562
Fold 1 AUC = 0.775562
FOLD 2
Training until validation scores don't improve for 300 rounds
[200]	valid_0's auc: 0.776429
[400]	valid_0's auc: 0.782476
[600]	valid_0's auc: 0.783422
[800]	valid_0's auc: 0.783387
Early stopping, best iteration is:
[642]	valid_0's auc: 0.783713
Fold 2 AUC = 0.783713
FOLD 3
Training until validation scores don't improve for 300 rounds
[200]	valid_0's auc: 0.768613
[400]	valid_0's auc: 0.774731
[600]	valid_0's auc: 0.775753
[800]	valid_0's auc: 0.775913
[1000]	valid_0's auc: 0.775679
Early stopping, best iteration is:
[718]	valid_0's auc: 0.775997
Fold 3 AUC = 0.775997
FOLD 4
Training until validation scores don't improve for 300 rounds
[200]	valid_0's auc: 0.774908
[400]	valid_0

In [52]:

# ============================================================
# OVERALL CV SCORE
# ============================================================

overall_auc = roc_auc_score(
    y,
    oof_preds
)

print("\n")
print("=" * 60)
print("CROSS VALIDATION RESULTS")
print("=" * 60)

for i, score in enumerate(cv_scores, 1):
    print(f"Fold {i}: {score:.6f}")

print("\n")

print(
    f"Mean CV AUC : {np.mean(cv_scores):.6f}"
)

print(
    f"Std CV AUC  : {np.std(cv_scores):.6f}"
)

print(
    f"OOF AUC      : {overall_auc:.6f}"
)





CROSS VALIDATION RESULTS
Fold 1: 0.775562
Fold 2: 0.783713
Fold 3: 0.775997
Fold 4: 0.782489
Fold 5: 0.774762


Mean CV AUC : 0.778505
Std CV AUC  : 0.003794
OOF AUC      : 0.778175


In [ ]:

# ============================================================
# FEATURE IMPORTANCE
# ============================================================

feature_importance_mean = (
    feature_importance
    .groupby('feature')['importance']
    .mean()
    .reset_index()
    .sort_values(
        by='importance',
        ascending=False
    )
)

print("\nTop 50 Features")

top_features=list(feature_importance_mean.head(50)['feature'])
top_features



Top 50 Features


['ORGANIZATION_TYPE',
 'credit_to_annuity',
 'goods_to_credit',
 'EXT_SOURCE_3',
 'EXT_SOURCE_1',
 'DAYS_LAST_PHONE_CHANGE',
 'AMT_ANNUITY',
 'ext_std',
 'OCCUPATION_TYPE',
 'annuity_to_income',
 'EXT_SOURCE_2',
 'DAYS_ID_PUBLISH',
 'ext_mean_x_employment',
 'id_publish_age_ratio',
 'ext_mean',
 'ext_min',
 'DAYS_REGISTRATION',
 'DAYS_BIRTH',
 'REGION_POPULATION_RELATIVE',
 'annuity_per_person',
 'registration_age_ratio',
 'income_to_days',
 'employment_age_ratio',
 'income_to_age',
 'ext_mean_x_age',
 'OWN_CAR_AGE',
 'credit_per_person',
 'ext_mul',
 'ext_max',
 'ext_med',
 'ext_mean_x_income',
 'income_to_price',
 'AMT_GOODS_PRICE',
 'total_consumer_credit_debt',
 'age_years',
 'AMT_CREDIT',
 'income_per_person',
 'total_credit_card_debt',
 'ACTIVE_BUREAU_B_TOTAL_TENURE_MIN',
 'credit_to_income',
 'ACTIVE_BUREAU_B_LEFT_TO_FULL_MEAN',
 'ACTIVE_BUREAU_DAYS_CREDIT_ENDDATE_MIN',
 'BUREAU_B_LEFT_TO_FULL_STD',
 'ACTIVE_BUREAU_B_LEFT_TO_FULL_MAX',
 'BUREAU_B_UPDATE_TO_AGE_STD',
 'DAYS_EMPLO

In [54]:

# ============================================================
# SAVE OOF
# ============================================================

oof_df = pd.DataFrame({
    ID_COL: train_df[ID_COL],
    TARGET_COL: y,
    'OOF_PRED': oof_preds
})

oof_df.to_parquet(
    '/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/model_data/all_feat_ver/lgb_baseline_oof.parquet',
    index=False
)

print(
    "\nSaved: lgb_baseline_oof.parquet"
)



Saved: lgb_baseline_oof.parquet


In [55]:

# ============================================================
# SAVE SUBMISSION
# ============================================================

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET_COL: test_preds
})

submission.to_csv(
    '/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/model_data/all_feat_ver/lgb_baseline_submission.csv',
    index=False
)

print(
    "Saved: lgb_baseline_submission.csv"
)


Saved: lgb_baseline_submission.csv


In [56]:

# ============================================================
# SAVE FEATURE IMPORTANCE
# ============================================================

feature_importance_mean.to_csv(
    '/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/model_data/all_feat_ver/lgb_feature_importance.csv',
    index=False
)

print(
    "Saved: lgb_feature_importance.csv"
)

Saved: lgb_feature_importance.csv
